# GIR 온실가스 배출량 기업-연도 패널 재검증

2020~2025년 GIR 원자료 6개 파일만 사용해 기업-연도 단위 패널을 다시 구축한다. 기존 분석의 집계 원칙을 유지하되, 파생파일이 원본 검증에 섞이지 않도록 입력 파일을 명시한다.

집계 원칙은 다음과 같다.

- 업체 행이 1개이면 업체 값을 사용한다.
- 업체 행이 여러 개이고 배출량이 같으면 한 번만 사용한다.
- 업체 행이 여러 개이고 배출량이 다르면 자동 합산하지 않고 제외한다.
- 업체 행이 없고 사업장이 1개이면 사업장 값을 사용한다.
- 업체 행이 없고 실질적으로 서로 다른 사업장이 여러 개이면 합산한다.
- 번호 또는 비고만 다른 실질 중복은 제거한다.

2023년의 **에스케이피유코어 주식회사**, **에이치디씨폴리올(주)**, **쿠팡로지스틱스서비스 유한회사**는 동일 사업장으로 보이는 행의 값이 거의 같지만 완전히 같지 않아 어느 값이 최종값인지 원자료만으로 판단하기 어렵다. 잘못된 이중 합산을 막기 위해 2023년 분석에서 제외한다. 반면 경원여객자동차처럼 실제로 서로 다른 사업장으로 확인되는 사례는 합산한다.

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

data_path = Path("../data")
years = list(range(2020, 2026))
gir_files = [data_path / f"gir_{year}.csv" for year in years]

missing_files = [str(path) for path in gir_files if not path.exists()]
assert not missing_files, f"원본 GIR 파일 누락: {missing_files}"

print("사용 원본 파일")
for path in gir_files:
    raw = pd.read_csv(path)
    print(path.name, "행 수:", len(raw), "대상년도:", sorted(raw["대상년도"].dropna().unique().tolist()))

사용 원본 파일
gir_2020.csv 행 수: 1061 대상년도: [2020]
gir_2021.csv 행 수: 1079 대상년도: [2021]
gir_2022.csv 행 수: 1102 대상년도: [2022]
gir_2023.csv 행 수: 1540 대상년도: [2023]
gir_2024.csv 행 수: 1167 대상년도: [2024]
gir_2025.csv 행 수: 1173 대상년도: [2025]


In [2]:
numeric_cols = ["온실가스 배출량(tCO₂eq)", "에너지 사용량(TJ)"]
ambiguous_2023 = [
    "에스케이피유코어 주식회사",
    "에이치디씨폴리올(주)",
    "쿠팡로지스틱스서비스 유한회사",
]

def make_company_data(df):
    df = df.copy()

    for col in numeric_cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(",", "", regex=False),
            errors="coerce",
        )

    # 번호·비고만 다른 실질 중복 제거
    core_cols = [col for col in df.columns if col not in ["번호", "비고"]]
    df = df.drop_duplicates(subset=core_cols, keep="first").copy()

    company_rows = []
    excluded_different_company_values = []

    for company, group in df.groupby("법인명", sort=True):
        company_level = group[group["지정구분"] == "업체"]

        if len(company_level) == 1:
            row = company_level.iloc[0].copy()
            row["집계방식"] = "업체 1개"

        elif len(company_level) > 1:
            ghg_values = company_level["온실가스 배출량(tCO₂eq)"].drop_duplicates()
            # NaN끼리도 동일 값으로 취급하되, 서로 다른 값이 섞이면 제외
            if len(ghg_values) == 1:
                row = company_level.iloc[0].copy()
                row["온실가스 배출량(tCO₂eq)"] = ghg_values.iloc[0]
                energy_values = company_level["에너지 사용량(TJ)"].drop_duplicates()
                if len(energy_values) == 1:
                    row["에너지 사용량(TJ)"] = energy_values.iloc[0]
                else:
                    row["에너지 사용량(TJ)"] = pd.NA
                row["집계방식"] = "업체 중복 동일값"
            else:
                excluded_different_company_values.append(company)
                continue

        else:
            sites = group[group["지정구분"] == "사업장"]
            if len(sites) == 0:
                continue
            if len(sites) == 1:
                row = sites.iloc[0].copy()
                row["집계방식"] = "단일 사업장"
            else:
                row = sites.iloc[0].copy()
                # 모두 결측이면 0이 되지 않도록 min_count=1 사용
                row["온실가스 배출량(tCO₂eq)"] = sites["온실가스 배출량(tCO₂eq)"].sum(min_count=1)
                row["에너지 사용량(TJ)"] = sites["에너지 사용량(TJ)"].sum(min_count=1)
                row["집계방식"] = f"사업장 {len(sites)}개 합산"

        company_rows.append(row)

    result = pd.DataFrame(company_rows)
    result.attrs["excluded_different_company_values"] = excluded_different_company_values
    return result

# 2023년 제외 대상 원자료 확인
display(
    pd.read_csv(data_path / "gir_2023.csv")
    .loc[lambda x: x["법인명"].isin(ambiguous_2023),
         ["법인명", "대상년도", "지정구분", "온실가스 배출량(tCO₂eq)", "에너지 사용량(TJ)", "비고"]]
    .sort_values(["법인명", "대상년도"])
)

,법인명,대상년도,지정구분,온실가스 배출량(tCO₂eq),에너지 사용량(TJ),비고
909,에스케이피유코어 주식회사,2023,사업장,22558,496,NaN
910,에스케이피유코어 주식회사,2023,사업장,22556,496,NaN
931,에이치디씨폴리올(주),2023,사업장,35086,588,NaN
932,에이치디씨폴리올(주),2023,사업장,35085,588,NaN
1311,쿠팡로지스틱스서비스 유한회사,2023,사업장,94808,1531,NaN
1312,쿠팡로지스틱스서비스 유한회사,2023,사업장,94998,1535,NaN


In [3]:
gir_company_list = []
aggregation_audit = []

for year, path in zip(years, gir_files):
    raw = pd.read_csv(path)
    if year == 2023:
        raw = raw[~raw["법인명"].isin(ambiguous_2023)].copy()

    company_df = make_company_data(raw)
    excluded = company_df.attrs.get("excluded_different_company_values", [])
    gir_company_list.append(company_df)
    aggregation_audit.append({
        "대상년도": year,
        "기업수": len(company_df),
        "서로다른_업체값으로_제외": len(excluded),
        "제외기업": ", ".join(excluded),
    })

gir_company = pd.concat(gir_company_list, ignore_index=True)
year_counts = gir_company.groupby("대상년도")["법인명"].nunique()
duplicate_company_year = gir_company.duplicated(["법인명", "대상년도"]).sum()
negative_ghg_count = (gir_company["온실가스 배출량(tCO₂eq)"] < 0).sum()
missing_ghg_count = gir_company["온실가스 배출량(tCO₂eq)"].isna().sum()

print("집계 감사")
display(pd.DataFrame(aggregation_audit))
print("최종 기업-연도:", len(gir_company))
print("기업명+대상년도 중복:", duplicate_company_year)
print("연도별 기업 수")
display(year_counts.rename("기업수").to_frame())
print("음수 배출량 수:", int(negative_ghg_count))
print("배출량 결측치 수:", int(missing_ghg_count))

assert duplicate_company_year == 0, "기업명+대상년도 중복이 남아 있습니다."
assert set(gir_company["대상년도"].dropna().astype(int)) == set(years)

output_path = data_path / "gir_company_2020_2025_reviewed.csv"
gir_company.to_csv(output_path, index=False, encoding="utf-8-sig")
print("저장:", output_path, gir_company.shape)

집계 감사


,대상년도,기업수,서로다른_업체값으로_제외,제외기업
0,2020,1060,0,
1,2021,1078,0,
2,2022,1095,0,
3,2023,1139,0,
4,2024,1166,0,
5,2025,1172,0,


최종 기업-연도: 6710
기업명+대상년도 중복: 0
연도별 기업 수


,기업수
대상년도,
2020,1060
2021,1078
2022,1095
2023,1139
2024,1166
2025,1172


음수 배출량 수: 0
배출량 결측치 수: 6
저장: ..\data\gir_company_2020_2025_reviewed.csv (6710, 11)


In [4]:
ghg_year_summary = (
    gir_company.groupby("대상년도")
    .agg(
        기업연도수=("법인명", "size"),
        고유기업수=("법인명", "nunique"),
        총배출량=("온실가스 배출량(tCO₂eq)", "sum"),
        평균배출량=("온실가스 배출량(tCO₂eq)", "mean"),
        중앙값배출량=("온실가스 배출량(tCO₂eq)", "median"),
        배출량결측=("온실가스 배출량(tCO₂eq)", lambda x: x.isna().sum()),
    )
)
display(ghg_year_summary.round(1))

,기업연도수,고유기업수,총배출량,평균배출량,중앙값배출량,배출량결측
대상년도,,,,,,
2020,1060,1060,587517090.0,555309.2,43958.5,2
2021,1078,1078,614837456.0,570879.7,51349.0,1
2022,1095,1095,596467313.0,545216.9,52848.5,1
2023,1139,1139,575863386.0,506031.1,53945.0,1
2024,1166,1166,565618230.0,485509.2,51427.0,1
2025,1172,1172,563672309.0,480949.1,50230.5,0


In [5]:
company_year_count = gir_company.groupby("법인명")["대상년도"].nunique()
common_companies = company_year_count[company_year_count == 6].index
gir_balanced = gir_company[gir_company["법인명"].isin(common_companies)].copy()

balanced_summary = (
    gir_balanced.groupby("대상년도")
    .agg(
        기업수=("법인명", "nunique"),
        총배출량=("온실가스 배출량(tCO₂eq)", "sum"),
        평균배출량=("온실가스 배출량(tCO₂eq)", "mean"),
        중앙값배출량=("온실가스 배출량(tCO₂eq)", "median"),
        배출량결측=("온실가스 배출량(tCO₂eq)", lambda x: x.isna().sum()),
    )
)

ghg_2020 = balanced_summary.loc[2020, "총배출량"]
ghg_2025 = balanced_summary.loc[2025, "총배출량"]
balanced_change_rate = (ghg_2025 - ghg_2020) / ghg_2020 * 100

company_change = (
    gir_balanced[gir_balanced["대상년도"].isin([2020, 2025])]
    .pivot(index="법인명", columns="대상년도", values="온실가스 배출량(tCO₂eq)")
)
valid_company_change = company_change.dropna(subset=[2020, 2025]).copy()
valid_company_change["변화량"] = valid_company_change[2025] - valid_company_change[2020]
valid_company_change["변화방향"] = valid_company_change["변화량"].map(
    lambda x: "감소" if x < 0 else ("증가" if x > 0 else "동일")
)

direction_order = ["감소", "증가", "동일"]
direction_counts = valid_company_change["변화방향"].value_counts().reindex(direction_order, fill_value=0)
direction_table = pd.DataFrame({
    "기업수": direction_counts,
    "비율(%)": (direction_counts / len(valid_company_change) * 100).round(1),
})

total_decrease = valid_company_change.loc[valid_company_change["변화량"] < 0, "변화량"].sum()
total_increase = valid_company_change.loc[valid_company_change["변화량"] > 0, "변화량"].sum()
net_change = valid_company_change["변화량"].sum()

top_reducers_table = (
    valid_company_change.sort_values("변화량").head(10).reset_index()
)
top_reducers_table["감소기여도(%)"] = (
    top_reducers_table["변화량"].abs() / abs(total_decrease) * 100
)
top10_share = top_reducers_table["감소기여도(%)"].sum()

print("Balanced panel 기업 수:", len(common_companies))
display(balanced_summary.round(1))
print(f"2020 총배출량: {ghg_2020:,.1f}")
print(f"2025 총배출량: {ghg_2025:,.1f}")
print(f"2020→2025 증감률: {balanced_change_rate:.2f}%")
print("2020·2025 배출량 모두 유효한 기업:", len(valid_company_change))
display(direction_table)
print(f"감소 기업 총 감소량: {total_decrease:,.1f}")
print(f"증가 기업 총 증가량: {total_increase:,.1f}")
print(f"순변화량: {net_change:,.1f}")
print(f"감소폭 상위 10개 기업의 감소 기여 비율: {top10_share:.2f}%")
display(top_reducers_table[["법인명", 2020, 2025, "변화량", "감소기여도(%)"]].round(1))

Balanced panel 기업 수: 742


,기업수,총배출량,평균배출량,중앙값배출량,배출량결측
대상년도,,,,,
2020,742,539114305.0,728532.8,64567.0,2
2021,742,554469340.0,748271.7,70000.0,1
2022,742,530139963.0,715438.5,67703.0,1
2023,742,509529527.0,687624.2,67451.0,1
2024,742,496442234.0,669962.5,67159.0,1
2025,742,488451329.0,658290.2,65174.0,0


2020 총배출량: 539,114,305.0
2025 총배출량: 488,451,329.0
2020→2025 증감률: -9.40%
2020·2025 배출량 모두 유효한 기업: 740


,기업수,비율(%)
변화방향,,
감소,379,51.2
증가,360,48.6
동일,1,0.1


감소 기업 총 감소량: -71,304,718.0
증가 기업 총 증가량: 20,516,203.0
순변화량: -50,788,515.0
감소폭 상위 10개 기업의 감소 기여 비율: 69.84%


대상년도,법인명,2020,2025,변화량,감소기여도(%)
0,한국남동발전,42521216.0,30835303.0,-11685913.0,16.4
1,한국동서발전 주식회사,34905756.0,26344859.0,-8560897.0,12.0
2,한국중부발전,34774881.0,28063372.0,-6711509.0,9.4
3,주식회사 포스코,75671169.0,69846022.0,-5825147.0,8.2
4,한국남부발전 주식회사,29007394.0,24407208.0,-4600186.0,6.5
5,(주)지에스동해전력,6797741.0,2693243.0,-4104498.0,5.8
6,한국서부발전,30422589.0,27282477.0,-3140112.0,4.4
7,주식회사 지에스이앤알,2057228.0,654.0,-2056574.0,2.9
8,삼표시멘트,5986309.0,4269882.0,-1716427.0,2.4
9,엘지디스플레이(주),4748207.0,3347075.0,-1401132.0,2.0


## 재검증 요약

- 입력은 `gir_2020.csv`부터 `gir_2025.csv`까지 원본 6개 파일로 고정했고, 파생파일은 제외했다.
- 번호·비고만 다른 실질 중복을 제거하고 기존 기업 단위 집계 원칙을 유지했다. 다중 사업장의 합계가 모두 결측일 때 0으로 바뀌지 않도록 결측치를 보존했다.
- 2023년의 판단 불가능한 근사 중복 3개 기업을 명시적으로 제외했다.
- 최종 기업-연도 행 수는 **6,710개**, 기업명+대상년도 중복은 **0개**, 음수 배출량은 **0개**, 배출량 결측치는 **6개**다.
- 6개 연도에 모두 존재하는 Balanced panel은 **742개 기업**이다. 이 가운데 2020년과 2025년 배출량이 모두 유효한 기업은 740개다.
- Balanced panel 총배출량은 2020년 **539,114,305 tCO₂eq**에서 2025년 **488,451,329 tCO₂eq**로 **9.40% 감소**했다.
- 기존 노트북에 저장돼 있던 파생파일의 6,713행과 비교하면, 2023년 제외 대상 3개 기업을 반영해 3행 감소했다. Balanced panel 742개와 약 9.4% 감소라는 핵심 결과는 유지됐다.